# BioVU EM-seq Data Processing
**Pershad & Zhao et al., Nature Medicine, 2025**

This notebook documents the processing of targeted enzymatic methyl-seq (EM-seq) data from peripheral whole blood
in the Vanderbilt BioVU cohort. Starting from per-sample CX_report files (Bismark output), we:
1. Filter CpG sites by coverage thresholds
2. Identify high-confidence CpG sites present across ≥80% of samples at ≥25× coverage
3. Construct a cohort-wide beta value matrix (samples × CpG sites)

The WDL pipeline on Terra (`wdl/optimized-matrix-generation.wdl`) automates steps 1–3 across all samples in parallel.
This notebook documents the underlying logic and is used for QC inspection of outputs.

**Sequencing:** Illumina NovaSeq 6000, 150 bp paired-end, ~30× mean depth  
**Panel:** Twist Biosciences hybrid capture, ~4 million CpG sites (GRCh38)

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

## 1. Per-sample CpG site extraction from CX_report files

CX_report files from Bismark have columns:  
`chr, position, strand, meth_count, unmeth_count, context, trinuc_context`

We extract CpG-context sites (`context == 'CG'`) with ≥10× total coverage,
and flag high-confidence sites at ≥25× coverage for cohort-level filtering.

This step is run per-sample in the WDL scatter task `extract_10x_sites`.

In [ ]:
%%bash
# Extract CpG sites from a single CX_report file.
# Outputs (per sample):
#   {sample}_filtered_data.txt  — CpGs at ≥10× coverage (site_id, chr, pos, strand, coverage, methylation%)
#   {sample}_sites_25x.txt     — site IDs at ≥25× coverage (used for cohort-level filtering)
#   {sample}_stats.txt         — coverage summary statistics

min_coverage=10
high_coverage=25
sample_name="SAMPLE_ID"  # replace with actual VANTAGE/GRID sample ID

zcat "${sample_name}.CX_report.txt.gz" | \
  awk -v min_cov="$min_coverage" -v high_cov="$high_coverage" -v sample="$sample_name" '
    BEGIN { OFS="\t" }
    $6 == "CG" && ($4 + $5) >= min_cov {
        coverage = $4 + $5
        beta_pct  = (coverage > 0) ? ($4 / coverage) * 100 : 0
        site_id   = $1 ":" $2

        # Full data for matrix construction
        print site_id, $1, $2, $3, coverage, beta_pct > (sample "_filtered_data.txt")

        # High-coverage site IDs for cohort filtering
        if (coverage >= high_cov) print site_id > (sample "_sites_25x.txt")
    }'

## 2. Identify cohort-level high-confidence CpG sites

We retain CpG sites present at ≥25× coverage in ≥80% of cohort samples.  
This stringent threshold ensures reliable beta values across the full cohort.

This step corresponds to the `find_common_sites` WDL task.

In [ ]:
%%bash
# Count how many samples have each CpG site at ≥25× coverage.
# Retain sites present in ≥80% of samples.

n_samples=$(ls *_sites_25x.txt | wc -l)
# Ceiling of 80% threshold
min_samples=$(echo "$n_samples * 0.8" | bc | awk '{print int($1) + ($1 > int($1))}')

echo "Total samples: $n_samples"
echo "Min samples required (80% threshold): $min_samples"

# Aggregate all site IDs across samples, count occurrences, filter
cat *_sites_25x.txt | sort | uniq -c | \
    awk -v min_s="$min_samples" '$1 >= min_s {print $2}' > selected_sites_25x.txt

echo "CpG sites retained: $(wc -l < selected_sites_25x.txt)"

## 3. Build beta value matrix

For each sample, we extract methylation beta values (= methylated reads / total coverage)
at the cohort-level selected sites, sort by site ID, and paste all samples into a single matrix.

Beta values are on the [0, 1] scale. Final matrix dimensions: ~2.5M sites × 560 samples.

In [ ]:
%%bash
# For each sample, extract methylation data at selected sites and sort by site ID.
for sample_file in *_filtered_data.txt; do
    sample=$(basename "$sample_file" _filtered_data.txt)
    # Retain only the cohort-level selected sites
    awk 'NR==FNR{sites[$1]=1; next} $1 in sites' \
        selected_sites_25x.txt "$sample_file" | sort -k1,1 > "${sample}_sorted.txt"
done

# Collect ordered sample names
mapfile -t samples < <(ls *_sorted.txt | sed 's/_sorted.txt//')

# Build header row
header="site_id\tchr\tpos\tstrand"
for s in "${samples[@]}"; do header+="\t${s}"; done
echo -e "$header" > beta_matrix.tsv

# Extract site coordinates from first sample
awk '{print $1"\t"$2"\t"$3"\t"$4}' "${samples[0]}_sorted.txt" > site_coords.txt

# Extract beta values (methylation% / 100) for each sample in parallel
for s in "${samples[@]}"; do
    awk '{printf "%.4f\n", $6/100}' "${s}_sorted.txt" > "beta_${s}.tmp"
done

# Paste coordinates and all beta columns, append to matrix
paste site_coords.txt beta_*.tmp >> beta_matrix.tsv
rm -f beta_*.tmp site_coords.txt

echo "Beta matrix written: beta_matrix.tsv"
echo "Dimensions: $(wc -l < beta_matrix.tsv) rows x $(head -1 beta_matrix.tsv | tr '\t' '\n' | wc -l) columns"

## 4. Merge batched Terra outputs and map to BioVU GRIDs

The WDL pipeline processes samples in batches on Terra. Here we merge the batch outputs
and rename columns from Vantage sequencing IDs to BioVU GRID identifiers.

In [ ]:
# Load per-batch beta matrices from Terra GCS output
# (GCS paths are placeholders; actual paths from your Terra workspace)
batch_gcs_paths = [
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/1-100/ewas_beta_values_matrix_1-100.tsv",
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/101-200/ewas_beta_values_matrix_101-200.tsv",
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/201-300/ewas_beta_values_matrix_201-300.tsv",
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/301-400/ewas_beta_values_matrix_301-400.tsv",
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/401-500/ewas_beta_values_matrix_401-500.tsv",
    "gs://fc-secure-WORKSPACE/biovu_chip_methylation_matrix_files/501-560/ewas_beta_values_matrix_501-560.tsv",
]

# Read and join all batches on the CpG site column ('genpos')
batches = [pd.read_table(p) for p in batch_gcs_paths]
combined_df = batches[0]
for b in batches[1:]:
    combined_df = combined_df.merge(b, on='genpos', how='inner')

print(f"Combined matrix (inner join): {combined_df.shape[0]:,} CpG sites x {combined_df.shape[1]} columns")

In [ ]:
# Load plate map: Vantage sequencing ID → BioVU GRID
plate_map = pd.read_table("12646-plate-map-grid.txt", header=None, names=['vantage_id', 'grid'])
plate_map = plate_map[plate_map['grid'] != 'Blank']  # remove empty wells
id_map = dict(zip(plate_map['vantage_id'], plate_map['grid']))
id_map['genpos'] = 'genpos'  # preserve site ID column name

# Rename columns and transpose to samples-as-rows format
combined_df = combined_df.rename(columns=id_map)
beta_matrix = combined_df.set_index('genpos').T  # samples (rows) × CpGs (columns)

print(f"Final beta matrix: {beta_matrix.shape[0]} samples x {beta_matrix.shape[1]:,} CpG sites")
beta_matrix.to_csv("biovu_chip_ewas_beta_matrix.txt", sep='\t')

## 5. QC: Inspect matrix and confirm data integrity

In [ ]:
beta_matrix = pd.read_csv("biovu_chip_ewas_beta_matrix.txt", sep='\t', index_col=0).astype('float32')

print(f"Matrix dimensions: {beta_matrix.shape[0]} samples × {beta_matrix.shape[1]:,} CpG sites")
print(f"Memory: {beta_matrix.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Beta value range: [{beta_matrix.values.min():.3f}, {beta_matrix.values.max():.3f}]")
print(f"Missing data fraction: {beta_matrix.isna().mean().mean():.4f}")

# Distribution of mean methylation per sample (should be ~0.5 for bulk blood)
sample_means = beta_matrix.mean(axis=1)
print(f"\nSample-level mean beta: {sample_means.mean():.3f} ± {sample_means.std():.3f}")

# Flag outlier samples (>3 SD from mean)
outliers = sample_means[np.abs(sample_means - sample_means.mean()) > 3 * sample_means.std()]
if len(outliers) > 0:
    print(f"Potential outlier samples (>3 SD from mean): {outliers.index.tolist()}")
else:
    print("No outlier samples detected.")

In [ ]:
# Prepare stratified metadata files for EWAS input (see wdl/run-logistic-ewas.wdl)
metadata_df = pd.read_table('metadata_biovu_chip_methylation.tsv')

# TET2 EWAS: exclude DNMT3A CHIP carriers
tet2_meta      = metadata_df[metadata_df['d3a'] == 0]
tet2_plof_meta = metadata_df[(metadata_df['d3a'] == 0) & (metadata_df['tet2_miss'] == 0)]
tet2_miss_meta  = metadata_df[(metadata_df['d3a'] == 0) & (metadata_df['tet2_fssg'] == 0)]

# DNMT3A EWAS: exclude TET2 CHIP carriers
d3a_meta       = metadata_df[metadata_df['tet2'] == 0]
d3a_r882_meta  = metadata_df[(metadata_df['tet2'] == 0) & (metadata_df['d3a_fssg'] == 0)]
d3a_nonr882_meta = metadata_df[(metadata_df['tet2'] == 0) & (metadata_df['d3a_r882'] == 0)]

# Save for WDL inputs
for df, name in [
    (tet2_meta,       'tet2_all'),
    (tet2_plof_meta,  'tet2_plof'),
    (tet2_miss_meta,  'tet2_miss'),
    (d3a_meta,        'd3a_all'),
    (d3a_r882_meta,   'd3a_r882'),
    (d3a_nonr882_meta,'d3a_nonr882'),
]:
    df.to_csv(f'{name}_metadata_for_ewas.tsv', sep='\t', index=False)
    print(f"{name}: N={len(df)} (cases: {df.get('chip_status', pd.Series()).sum()})")